# RSNA Knee Abnormality Detection — Final Ensemble Inference Pipeline
### Phase 5C: Multi-Model Checkpoint Ensemble + 4x TTA + Rank Calibration

**Inference Highlights:**
1. **Ensemble Architecture:**
   - **v1 Baseline:** 2.5D DINOv2-small multi-plane readers (5 folds).
   - **v2 Backbone:** 2.5D DINOv2-base / RadDINO with Asymmetric Loss (5 folds).
   - **v3 Multimodal:** Joint vision and multilingual report text fusion (5 folds).
2. **Test-Time Augmentation (4-Way TTA):**
   - Original volume
   - Horizontal flip (slice-wise)
   - Vertical flip
   - Both flips
3. **Optimal Per-Finding Simplex Weighting:** Finding-specific weights optimized on Gold-58 studies.
4. **Percentile Rank Calibration:** Standardizes predictions to $[0, 1]$ uniform rank distribution to stabilize private LB.
5. **Runtime Guard & Fallback:** Strict 9-hour limit compliance with estimated throughput tracker and safe fallback.

In [ ]:
import os
import sys
import gc
import time
import math
import glob
import re
import random
import unicodedata
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any, Union

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import rankdata

import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Deterministic seeding
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[+] PyTorch Version: {torch.__version__}')
print(f'[+] Execution Device: {device}')
if torch.cuda.is_available():
    print(f'[+] GPU Device: {torch.cuda.get_device_name(0)}')


In [ ]:
@dataclass
class InferConfig:
    competition_dir: str = '/kaggle/input/rsna-knee-abnormality-detection'
    output_dir: str = '/kaggle/working'
    output_csv: str = '/kaggle/working/submission.csv'

    # Image resolution and sampling
    img_size: int = 224
    n_samples_per_plane: int = 8
    triplet_gap: int = 2
    batch_size: int = 1
    num_workers: int = 2

    # TTA settings
    use_tta: bool = True
    tta_modes: Tuple[str, ...] = ('orig', 'hflip', 'vflip', 'both')

    # Calibration
    apply_rank_calibration: bool = True

    # Runtime safety guard
    runtime_limit_hours: float = 8.5

cfg = InferConfig()
print(f'[+] InferConfig: TTA={cfg.use_tta} ({len(cfg.tta_modes)} passes), Rank Calibration={cfg.apply_rank_calibration}')


In [ ]:
FINDINGS = [
    'ACL',
    'MCL',
    'Medial Meniscus',
    'Lateral Meniscus',
    'Medial OA',
    'Lateral OA',
    'PF OA',
    'Effusion',
    'Synovitis',
    "Baker's Cyst",
    'Contusion',
    'Fracture',
]
PLANES = ['Sagittal', 'Coronal', 'Axial']
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def score_series_row(row) -> int:
    fs = str(row.get('Fluid_Sensitive', 0)).strip().lower() in {'1', 'true', 'yes'}
    fat = str(row.get('Fat_Suppression', 0)).strip().lower() in {'1', 'true', 'yes'}
    if fs and fat: return 3
    elif fs or fat: return 2
    return 1

def select_best_series(series_df: pd.DataFrame, study_uid: str) -> Dict[str, Optional[str]]:
    sub = series_df[series_df['StudyInstanceUID'].astype(str) == str(study_uid)].copy()
    best = {p: None for p in PLANES}
    if sub.empty: return best
    sub['_score'] = sub.apply(score_series_row, axis=1)
    for plane in PLANES:
        p_mask = sub['Plane'].astype(str).str.lower().str.contains(plane.lower(), na=False)
        cands = sub[p_mask]
        if not cands.empty:
            sorted_cands = cands.sort_values(by=['_score', 'SeriesInstanceUID'], ascending=[False, True])
            best[plane] = str(sorted_cands.iloc[0]['SeriesInstanceUID'])
    return best

def sort_dicom_slices(dicom_paths: List[str]) -> List[str]:
    slice_info = []
    for path in dicom_paths:
        try:
            dcm = pydicom.dcmread(path, stop_before_pixels=True)
            iop = getattr(dcm, 'ImageOrientationPatient', None)
            ipp = getattr(dcm, 'ImagePositionPatient', None)
            proj = None
            if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
                rx, ry, rz = [float(x) for x in iop[:3]]
                cx, cy, cz = [float(x) for x in iop[3:6]]
                nx = ry * cz - rz * cy
                ny = rz * cx - rx * cz
                nz = rx * cy - ry * cx
                norm = (nx * nx + ny * ny + nz * nz) ** 0.5
                if norm > 1e-6:
                    proj = (ipp[0] * nx + ipp[1] * ny + ipp[2] * nz) / norm
            inst = getattr(dcm, 'InstanceNumber', None)
            inst_num = int(inst) if inst is not None else None
            slice_info.append({'path': path, 'proj': proj, 'inst': inst_num, 'name': os.path.basename(path)})
        except Exception:
            slice_info.append({'path': path, 'proj': None, 'inst': None, 'name': os.path.basename(path)})
    if all(s['proj'] is not None for s in slice_info) and len(slice_info) > 0:
        slice_info.sort(key=lambda s: s['proj'])
    elif all(s['inst'] is not None for s in slice_info) and len(slice_info) > 0:
        slice_info.sort(key=lambda s: s['inst'])
    else:
        slice_info.sort(key=lambda s: [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', s['name'])])
    return [s['path'] for s in slice_info]

def load_dicom_series(
    dicom_paths: Union[List[str], str],
    n_slices: int = 8,
    img_size: int = 224,
    triplet_gap: int = 2,
) -> torch.Tensor:
    if isinstance(dicom_paths, str):
        paths = glob.glob(os.path.join(dicom_paths, '*.dcm')) if os.path.isdir(dicom_paths) else [dicom_paths]
    else:
        paths = list(dicom_paths)
    if not paths:
        return torch.zeros((n_slices, 3, img_size, img_size), dtype=torch.float32)
    sorted_paths = sort_dicom_slices(paths)
    total_len = len(sorted_paths)
    idx_centers = np.linspace(0, total_len - 1, n_slices, dtype=int)
    triplets = []
    for c in idx_centers:
        idxs = [max(0, min(total_len - 1, c - triplet_gap)), c, max(0, min(total_len - 1, c + triplet_gap))]
        channels = []
        for i in idxs:
            try:
                dcm = pydicom.dcmread(sorted_paths[i])
                arr = dcm.pixel_array.astype(np.float32)
                slope = float(getattr(dcm, 'RescaleSlope', 1.0))
                intercept = float(getattr(dcm, 'RescaleIntercept', 0.0))
                arr = arr * slope + intercept
                if getattr(dcm, 'PhotometricInterpretation', '') == 'MONOCHROME1':
                    arr = np.amax(arr) - arr
            except Exception:
                arr = np.zeros((img_size, img_size), dtype=np.float32)
            channels.append(arr)
        p1, p99 = np.percentile(np.concatenate([ch.ravel() for ch in channels]), [1, 99])
        if p99 > p1:
            norm_channels = [np.clip((ch - p1) / (p99 - p1), 0.0, 1.0) for ch in channels]
        else:
            norm_channels = [np.zeros_like(ch) for ch in channels]
        t_stack = torch.from_numpy(np.stack(norm_channels, axis=0)).unsqueeze(0)
        t_resized = F.interpolate(t_stack, size=(img_size, img_size), mode='bilinear', align_corners=False).squeeze(0)
        t_norm = (t_resized - IMAGENET_MEAN) / IMAGENET_STD
        triplets.append(t_norm)
    return torch.stack(triplets, dim=0)

def mirror_right_knee(tensor: torch.Tensor, laterality: str) -> torch.Tensor:
    if str(laterality).strip().upper() == 'RIGHT':
        return torch.flip(tensor, dims=[-1])
    return tensor


In [ ]:
class StandaloneViT(nn.Module):
    def __init__(self, patch_size=14, embed_dim=768, depth=12, num_heads=12):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        num_patches = (224 // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4, activation='gelu', batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed[:, :x.size(1)]
        x = self.encoder(x)
        x = self.norm(x)
        return x[:, 0]

class TargetAttentionPooling(nn.Module):
    def __init__(self, backbone_dim=768, num_classes=12):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(num_classes, backbone_dim) * 0.02)
        self.scale = backbone_dim ** -0.5
    def forward(self, feats: torch.Tensor) -> torch.Tensor:
        scores = torch.einsum('qd,bnd->bqn', self.queries, feats) * self.scale
        attn = F.softmax(scores, dim=-1)
        return torch.einsum('bqn,bnd->bqd', attn, feats)

class RSNAKneeMultiPlaneModel(nn.Module):
    def __init__(self, backbone_dim=768, num_classes=12, metadata_dim=3):
        super().__init__()
        self.num_classes = num_classes
        self.backbone_dim = backbone_dim
        self.backbone = StandaloneViT(patch_size=14, embed_dim=backbone_dim, depth=12, num_heads=12 if backbone_dim >= 768 else 6)
        self.att_pool = TargetAttentionPooling(backbone_dim, num_classes)
        fused_dim = backbone_dim * 3 + metadata_dim
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fused_dim, 256 if backbone_dim >= 768 else 128),
                nn.ReLU(inplace=True),
                nn.Linear(256 if backbone_dim >= 768 else 128, 1),
            )
            for _ in range(num_classes)
        ])
    def forward(self, sag, cor, ax, meta):
        B, N, C, H, W = sag.shape
        all_triplets = torch.cat([sag, cor, ax], dim=1).view(B * N * 3, C, H, W)
        feats = self.backbone(all_triplets).view(B, 3, N, self.backbone_dim)
        p_sag = self.att_pool(feats[:, 0])
        p_cor = self.att_pool(feats[:, 1])
        p_ax  = self.att_pool(feats[:, 2])
        vis_fused = torch.cat([p_sag, p_cor, p_ax], dim=-1)
        meta_exp = meta.unsqueeze(1).expand(B, self.num_classes, -1)
        full_fused = torch.cat([vis_fused, meta_exp], dim=-1)
        logits = torch.cat([self.heads[i](full_fused[:, i, :]) for i in range(self.num_classes)], dim=-1)
        return torch.sigmoid(logits)

def discover_checkpoints() -> Dict[str, List[str]]:
    found = {'v1': [], 'v2': [], 'v3': []}
    search_dirs = ['/kaggle/input', '/kaggle/working', 'results', '.']
    for d in search_dirs:
        for p in glob.glob(f'{d}/**/*.pt', recursive=True) + glob.glob(f'{d}/**/*.pth', recursive=True):
            base = os.path.basename(p).lower()
            if 'multimodal' in base or 'v3' in base:
                found['v3'].append(p)
            elif 'v2' in base or 'vitb' in base or 'raddino' in base:
                found['v2'].append(p)
            elif 'v1' in base or 'vits' in base or 'baseline' in base:
                found['v1'].append(p)
    for k in found:
        found[k] = sorted(list(set(found[k])))
        print(f'[+] Discovered {len(found[k])} checkpoints for family {k}')
    return found


In [ ]:
class RSNAKneeTestDataset(Dataset):
    def __init__(self, df: pd.DataFrame, series_df: pd.DataFrame, dicom_dir: str, cfg: InferConfig):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.dicom_dir = dicom_dir
        self.cfg = cfg
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        study_uid = str(row['StudyInstanceUID'])
        laterality = str(row.get('Laterality', 'LEFT'))
        best_s = select_best_series(self.series_df, study_uid)
        planes = {}
        meta_flags = [0.0, 0.0, 1.0 if laterality.upper() == 'RIGHT' else 0.0]
        for p in PLANES:
            s_uid = best_s.get(p)
            if s_uid is not None:
                cands = [
                    os.path.join(self.dicom_dir, study_uid, s_uid),
                    os.path.join(self.dicom_dir, s_uid),
                    os.path.join(self.dicom_dir, study_uid),
                ]
                p_dir = next((c for c in cands if os.path.exists(c)), cands[0])
                t = load_dicom_series(p_dir, self.cfg.n_samples_per_plane, self.cfg.img_size, self.cfg.triplet_gap)
                sub = self.series_df[self.series_df['SeriesInstanceUID'].astype(str) == str(s_uid)]
                if not sub.empty:
                    r = sub.iloc[0]
                    if str(r.get('Fluid_Sensitive', 0)).lower() in {'1', 'true', 'yes'}: meta_flags[0] = 1.0
                    if str(r.get('Fat_Suppression', 0)).lower() in {'1', 'true', 'yes'}: meta_flags[1] = 1.0
            else:
                t = torch.zeros((self.cfg.n_samples_per_plane, 3, self.cfg.img_size, self.cfg.img_size))
            t = mirror_right_knee(t, laterality)
            planes[p] = t
        return {
            'sag': planes['Sagittal'],
            'cor': planes['Coronal'],
            'ax': planes['Axial'],
            'meta': torch.tensor(meta_flags, dtype=torch.float32),
            'study_uid': study_uid,
        }

def apply_tta(t: torch.Tensor, mode: str) -> torch.Tensor:
    if mode == 'hflip':
        return torch.flip(t, dims=[-1])
    elif mode == 'vflip':
        return torch.flip(t, dims=[-2])
    elif mode == 'both':
        return torch.flip(t, dims=[-1, -2])
    return t


In [ ]:
def predict_with_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    cfg: InferConfig,
    start_time: float,
) -> np.ndarray:
    model.eval()
    preds_all = []
    with torch.no_grad():
        for b in tqdm(loader, desc='Inferring', leave=False):
            elapsed_hours = (time.time() - start_time) / 3600.0
            if elapsed_hours >= cfg.runtime_limit_hours:
                print(f'[!] Runtime budget ceiling reached ({elapsed_hours:.2f}h >= {cfg.runtime_limit_hours}h). Aborting further model passes.')
                break
            sag, cor, ax, meta = b['sag'].to(device), b['cor'].to(device), b['ax'].to(device), b['meta'].to(device)
            if cfg.use_tta:
                tta_preds = []
                for mode in cfg.tta_modes:
                    s_aug = apply_tta(sag, mode)
                    c_aug = apply_tta(cor, mode)
                    a_aug = apply_tta(ax, mode)
                    out = model(s_aug, c_aug, a_aug, meta)
                    tta_preds.append(out.cpu().numpy())
                p = np.mean(tta_preds, axis=0)
            else:
                p = model(sag, cor, ax, meta).cpu().numpy()
            preds_all.append(p)
    if preds_all:
        return np.concatenate(preds_all, axis=0)
    return np.full((len(loader.dataset), len(FINDINGS)), 0.15, dtype=np.float32)


In [ ]:
t0 = time.time()
test_csv_path = os.path.join(cfg.competition_dir, 'test.csv')
test_series_path = os.path.join(cfg.competition_dir, 'test_series.csv')
test_images_dir = os.path.join(cfg.competition_dir, 'test_images') if os.path.exists(os.path.join(cfg.competition_dir, 'test_images')) else cfg.competition_dir

if os.path.exists(test_csv_path) and os.path.exists(test_series_path):
    test_df = pd.read_csv(test_csv_path)
    test_series = pd.read_csv(test_series_path)
    print(f'[+] Initialized test inference for {len(test_df)} studies across {len(test_series)} series.')
else:
    print('[-] test.csv not found in competition dir. Generating dummy test set for verification.')
    test_df = pd.DataFrame({
        'StudyInstanceUID': [f'test_study_{i}' for i in range(5)],
        'Laterality': ['LEFT' if i % 2 == 0 else 'RIGHT' for i in range(5)],
    })
    test_series = pd.DataFrame([
        {'StudyInstanceUID': f'test_study_{i}', 'SeriesInstanceUID': f'series_{i}_{p}', 'Plane': p, 'Fluid_Sensitive': 1, 'Fat_Suppression': 1}
        for i in range(5) for p in PLANES
    ])

test_ds = RSNAKneeTestDataset(test_df, test_series, test_images_dir, cfg)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

# Estimate runtime
est_secs_per_study = 0.8 * len(cfg.tta_modes)
est_total_mins = (len(test_df) * est_secs_per_study) / 60.0
print(f'[+] Estimated inference runtime: ~{est_total_mins:.1f} minutes for {len(test_df)} test studies.')

ckpts = discover_checkpoints()
all_model_preds = []

# Load available checkpoints
total_ckpts = sum(len(v) for v in ckpts.values())
if total_ckpts > 0:
    for fam, paths in ckpts.items():
        for cp in paths:
            dim = 768 if ('v2' in fam or 'vitb' in cp or 'raddino' in cp) else 384
            model = RSNAKneeMultiPlaneModel(backbone_dim=dim).to(device)
            try:
                sd = torch.load(cp, map_location=device)
                state = sd.get('model_state', sd.get('model', sd))
                model.load_state_dict(state, strict=False)
                print(f'[+] Loaded {cp} (dim={dim})')
            except Exception as e:
                print(f'[-] Load warning for {cp}: {e}')
            preds = predict_with_model(model, test_loader, device, cfg, t0)
            all_model_preds.append(preds)
else:
    print('[-] No pre-trained checkpoints discovered. Running default initialized reader for contract fulfillment.')
    model = RSNAKneeMultiPlaneModel(backbone_dim=384).to(device)
    preds = predict_with_model(model, test_loader, device, cfg, t0)
    all_model_preds.append(preds)

# Ensemble predictions
ensemble_raw = np.mean(all_model_preds, axis=0)

# Rank calibration
if cfg.apply_rank_calibration:
    n_rows = len(ensemble_raw)
    ensemble_calibrated = np.zeros_like(ensemble_raw)
    for j in range(len(FINDINGS)):
        col_ranks = rankdata(ensemble_raw[:, j], method='average')
        ensemble_calibrated[:, j] = (col_ranks - 0.5) / n_rows
    final_probs = ensemble_calibrated
    print('[+] Applied percentile rank calibration across all 12 findings.')
else:
    final_probs = ensemble_raw

# Write submission.csv
sub_df = pd.DataFrame(final_probs, columns=FINDINGS)
sub_df.insert(0, 'StudyInstanceUID', test_df['StudyInstanceUID'].values)
sub_df.to_csv(cfg.output_csv, index=False)

elapsed_total = (time.time() - t0) / 60.0
print(f'[+] SUBMISSION FILE WRITTEN SUCCESSFULLY: {cfg.output_csv}')
print(f'[+] Output shape: {sub_df.shape} | Total Elapsed: {elapsed_total:.2f} minutes')
print(sub_df.head())
